# 04. Tool Use y Function Calling

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 90 minutos  
**Prerequisitos:** [01-03: Intro, Prompting, ReAct](01-intro-llm-agents.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Definir herramientas (tools) con schemas formales para LLMs
- Usar Function Calling APIs de OpenAI y Anthropic
- Implementar estrategias de selección de herramientas
- Integrar APIs externas (clima, búsqueda web, bases de datos)
- Aplicar sandboxing y validación para ejecución segura
- Componer herramientas para tareas complejas

## 1. Motivación: Extender Capacidades del LLM

### El Problema: LLMs Tienen Limitaciones Inherentes

Un LLM puro **no puede**:
- Acceder a datos en tiempo real (clima, noticias, precios)
- Realizar cálculos precisos de grandes números
- Ejecutar código o scripts
- Interactuar con bases de datos
- Enviar emails o hacer llamadas API
- Modificar archivos o sistemas

### La Solución: Function Calling / Tool Use

**Function Calling** permite que LLMs:
1. **Identifiquen** cuándo necesitan una herramienta
2. **Seleccionen** la herramienta apropiada
3. **Generen** argumentos correctamente formateados
4. **Interpreten** resultados de la ejecución

**Ejemplo:**
```python
User: "¿Cuál es el clima en Madrid ahora?"

LLM: {
  "function_name": "get_weather",
  "arguments": {
    "location": "Madrid, Spain",
    "unit": "celsius"
  }
}

Tool Execution: {"temp": 18, "condition": "sunny", ...}

LLM: "Actualmente en Madrid hace 18°C y está soleado."
```

### Pregunta Guía

**Al final responderemos:**
*¿Cómo podemos dar a LLMs acceso seguro y confiable a herramientas externas sin comprometer seguridad?*

## 2. Intuición Visual: Anatomía de una Herramienta

### Componentes de una Herramienta

```
┌─────────────────────────────────────────────────────────┐
│                   TOOL DEFINITION                       │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  1. NAME                                                │
│     ├─ Identificador único                              │
│     └─ Ej: "get_weather", "search_web"                 │
│                                                         │
│  2. DESCRIPTION                                         │
│     ├─ Qué hace la herramienta                          │
│     ├─ Cuándo usarla                                    │
│     └─ Ej: "Obtiene clima actual de una ubicación"     │
│                                                         │
│  3. PARAMETERS (JSON Schema)                            │
│     ├─ Inputs requeridos                                │
│     ├─ Tipos de datos                                   │
│     ├─ Validaciones                                     │
│     └─ Ej: {"location": "string", "unit": "enum"}      │
│                                                         │
│  4. RETURNS                                             │
│     ├─ Formato de output                                │
│     └─ Ej: {"temp": float, "condition": string}        │
│                                                         │
│  5. IMPLEMENTATION                                      │
│     ├─ Función ejecutable                               │
│     ├─ Manejo de errores                                │
│     └─ Sandboxing (si es necesario)                     │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### Flow de Function Calling

```
User Query
    ↓
┌─────────────────┐
│  LLM + Tools    │
│  Definitions    │
└─────────────────┘
    ↓
¿Necesita tool?
    ├─→ NO → Direct Response
    │
    └─→ SÍ
         ↓
    Tool Selection
         ↓
    Arguments Generation (JSON)
         ↓
    ┌─────────────────┐
    │  Validation     │  ← Verificar schema
    └─────────────────┘
         ↓
    ┌─────────────────┐
    │  Execution      │  ← Run tool safely
    └─────────────────┘
         ↓
    Tool Result
         ↓
    ┌─────────────────┐
    │  LLM + Result   │  ← Generate final response
    └─────────────────┘
         ↓
    Final Answer
```

In [ ]:
# Instalación
# !pip install openai anthropic requests python-dotenv pydantic

import os
import json
import requests
from typing import List, Dict, Optional, Any, Callable
from dataclasses import dataclass, field
from enum import Enum
import re

try:
    from pydantic import BaseModel, Field, ValidationError
    PYDANTIC_AVAILABLE = True
except ImportError:
    PYDANTIC_AVAILABLE = False
    print("⚠️  Pydantic no disponible")

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI no disponible")

from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas")

## 3. Fundamentos: JSON Schema para Tools

### JSON Schema Standard

OpenAI y Anthropic usan JSON Schema para definir herramientas:

$$
\begin{align}
\text{Tool} &= (\text{name}, \text{description}, \text{parameters}) \tag{1} \\
\text{parameters} &= \text{JSONSchema} \tag{2} \\
\text{JSONSchema} &= \{\text{type}, \text{properties}, \text{required}\} \tag{3}
\end{align}
$$

### Ejemplo Formal

```json
{
  "name": "get_weather",
  "description": "Obtiene el clima actual de una ubicación",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string",
        "description": "Ciudad y país, ej: 'Paris, France'"
      },
      "unit": {
        "type": "string",
        "enum": ["celsius", "fahrenheit"],
        "description": "Unidad de temperatura"
      }
    },
    "required": ["location"]
  }
}
```

### LLM Output Format

$$
\text{LLM}(\text{query}, \text{tools}) \to \begin{cases}
\text{function\_call}: \{\text{name}, \text{arguments}\} & \text{si necesita tool} \\
\text{content}: \text{string} & \text{respuesta directa}
\end{cases} \tag{4}
$$

## 4. Implementación: Tool Registry & Executor

In [ ]:
@dataclass
class Tool:
    """Definición de una herramienta"""
    name: str
    description: str
    parameters: Dict[str, Any]
    function: Callable
    
    def to_openai_format(self) -> Dict:
        """Convierte a formato OpenAI Function Calling"""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters
            }
        }
    
    def validate_args(self, arguments: Dict) -> Tuple[bool, Optional[str]]:
        """Valida argumentos contra schema"""
        required = self.parameters.get("required", [])
        
        # Verificar required fields
        for field in required:
            if field not in arguments:
                return False, f"Campo requerido faltante: {field}"
        
        # Verificar tipos (simplificado)
        properties = self.parameters.get("properties", {})
        for key, value in arguments.items():
            if key in properties:
                expected_type = properties[key].get("type")
                # Type checking básico
                if expected_type == "string" and not isinstance(value, str):
                    return False, f"{key} debe ser string"
                elif expected_type == "number" and not isinstance(value, (int, float)):
                    return False, f"{key} debe ser number"
        
        return True, None
    
    def execute(self, arguments: Dict) -> Any:
        """Ejecuta la herramienta con argumentos validados"""
        # Validar
        valid, error = self.validate_args(arguments)
        if not valid:
            return {"error": error}
        
        # Ejecutar
        try:
            result = self.function(**arguments)
            return result
        except Exception as e:
            return {"error": str(e)}

print("✅ Tool class definida")

In [ ]:
# Definir herramientas de ejemplo

def get_weather(location: str, unit: str = "celsius") -> Dict:
    """
    Obtiene clima actual (simulado para demo).
    En producción, usar API real como OpenWeatherMap.
    """
    # Simulación
    weather_db = {
        "madrid": {"temp": 18, "condition": "sunny", "humidity": 45},
        "london": {"temp": 12, "condition": "rainy", "humidity": 75},
        "new york": {"temp": 22, "condition": "cloudy", "humidity": 60},
    }
    
    location_key = location.lower().split(",")[0].strip()
    
    if location_key in weather_db:
        data = weather_db[location_key].copy()
        if unit == "fahrenheit":
            data["temp"] = data["temp"] * 9/5 + 32
        data["unit"] = unit
        return data
    else:
        return {"error": f"Clima no disponible para {location}"}

def calculate(expression: str) -> Dict:
    """
    Calcula expresión matemática.
    ADVERTENCIA: eval() es peligroso en producción.
    """
    try:
        # Sanitizar input (básico - mejorar en producción)
        allowed_chars = set("0123456789+-*/().%\s")
        if not all(c in allowed_chars for c in expression):
            return {"error": "Expresión contiene caracteres no permitidos"}
        
        result = eval(expression)
        return {"result": result, "expression": expression}
    except Exception as e:
        return {"error": str(e)}

def search_web(query: str, num_results: int = 3) -> Dict:
    """
    Búsqueda web simulada.
    En producción, usar API de Google, Bing, o SerpAPI.
    """
    # Simulación
    return {
        "query": query,
        "results": [
            {"title": f"Resultado {i+1} para '{query}'", "url": f"https://example.com/{i}"}
            for i in range(num_results)
        ]
    }

# Crear Tools
weather_tool = Tool(
    name="get_weather",
    description="Obtiene el clima actual de una ubicación específica",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "Ciudad y país, ej: 'Madrid, Spain'"
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Unidad de temperatura"
            }
        },
        "required": ["location"]
    },
    function=get_weather
)

calculator_tool = Tool(
    name="calculate",
    description="Calcula expresiones matemáticas. Soporta +, -, *, /, (), %",
    parameters={
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "Expresión matemática a calcular, ej: '2 + 2' o '(10 * 5) / 2'"
            }
        },
        "required": ["expression"]
    },
    function=calculate
)

search_tool = Tool(
    name="search_web",
    description="Busca información en la web",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Query de búsqueda"
            },
            "num_results": {
                "type": "number",
                "description": "Número de resultados a retornar"
            }
        },
        "required": ["query"]
    },
    function=search_web
)

print("✅ Herramientas definidas:")
print(f"  - {weather_tool.name}")
print(f"  - {calculator_tool.name}")
print(f"  - {search_tool.name}")

### Tool-Enabled Agent

In [ ]:
class FunctionCallingAgent:
    """
    Agente con capacidad de Function Calling.
    Soporta OpenAI function calling format.
    """
    
    def __init__(self, tools: List[Tool], llm_backend: str = "simulated"):
        self.tools = {tool.name: tool for tool in tools}
        self.llm_backend = llm_backend
        self.conversation_history = []
        
        if llm_backend == "openai" and OPENAI_AVAILABLE:
            self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            self.model = "gpt-4-turbo-preview"
        else:
            self.client = None
    
    def _call_llm(self, messages: List[Dict], tools: List[Dict]) -> Dict:
        """Llama al LLM con tools disponibles"""
        if self.llm_backend == "openai" and self.client:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=tools,
                tool_choice="auto"
            )
            
            message = response.choices[0].message
            
            # Convertir a formato estándar
            result = {"content": message.content}
            
            if message.tool_calls:
                result["tool_calls"] = [
                    {
                        "id": tc.id,
                        "name": tc.function.name,
                        "arguments": json.loads(tc.function.arguments)
                    }
                    for tc in message.tool_calls
                ]
            
            return result
        else:
            return self._simulated_llm(messages)
    
    def _simulated_llm(self, messages: List[Dict]) -> Dict:
        """Simulación de LLM con function calling"""
        last_message = messages[-1]["content"].lower()
        
        # Detectar intenciones
        if "clima" in last_message or "weather" in last_message or "temperatura" in last_message:
            # Extraer ubicación (simplificado)
            location = "Madrid"
            if "london" in last_message or "londres" in last_message:
                location = "London"
            elif "new york" in last_message or "nueva york" in last_message:
                location = "New York"
            
            return {
                "tool_calls": [{
                    "id": "call_1",
                    "name": "get_weather",
                    "arguments": {"location": location, "unit": "celsius"}
                }]
            }
        
        elif "calcul" in last_message or "cuánto es" in last_message or any(op in last_message for op in ["+", "-", "*", "/"]):
            # Extraer expresión
            match = re.search(r'[\d+\-*/().\s]+', last_message)
            expr = match.group(0).strip() if match else "2+2"
            
            return {
                "tool_calls": [{
                    "id": "call_1",
                    "name": "calculate",
                    "arguments": {"expression": expr}
                }]
            }
        
        elif "busca" in last_message or "search" in last_message or "información" in last_message:
            # Extraer query
            query = last_message.split("sobre")[-1].strip() if "sobre" in last_message else "información"
            
            return {
                "tool_calls": [{
                    "id": "call_1",
                    "name": "search_web",
                    "arguments": {"query": query, "num_results": 3}
                }]
            }
        
        # Respuesta directa
        return {"content": "¿En qué puedo ayudarte? Puedo consultar clima, hacer cálculos, o buscar información."}
    
    def run(self, query: str, max_iterations: int = 5, verbose: bool = True) -> str:
        """
        Ejecuta el agente con function calling.
        """
        if verbose:
            print(f"\n{'='*70}")
            print(f"🤖 Function Calling Agent")
            print(f"{'='*70}")
            print(f"\n❓ Query: {query}\n")
        
        # Inicializar conversación
        messages = [
            {"role": "system", "content": "Eres un asistente útil con acceso a herramientas."},
            {"role": "user", "content": query}
        ]
        
        # Convertir tools a formato OpenAI
        tools_format = [tool.to_openai_format() for tool in self.tools.values()]
        
        for iteration in range(max_iterations):
            if verbose:
                print(f"\n--- Iteración {iteration + 1} ---")
            
            # Llamar LLM
            response = self._call_llm(messages, tools_format)
            
            # Si hay tool calls, ejecutar
            if "tool_calls" in response:
                for tool_call in response["tool_calls"]:
                    tool_name = tool_call["name"]
                    arguments = tool_call["arguments"]
                    
                    if verbose:
                        print(f"\n🔧 Llamando herramienta: {tool_name}")
                        print(f"📥 Argumentos: {json.dumps(arguments, indent=2)}")
                    
                    # Ejecutar tool
                    if tool_name in self.tools:
                        result = self.tools[tool_name].execute(arguments)
                        
                        if verbose:
                            print(f"📤 Resultado: {json.dumps(result, indent=2)}")
                        
                        # Agregar resultado a mensajes
                        messages.append({
                            "role": "function",
                            "name": tool_name,
                            "content": json.dumps(result)
                        })
                    else:
                        if verbose:
                            print(f"❌ Herramienta no encontrada: {tool_name}")
                
                # Continuar loop para generar respuesta final
                continue
            
            # Si no hay tool calls, retornar respuesta
            if response.get("content"):
                if verbose:
                    print(f"\n✅ Respuesta Final:\n{response['content']}")
                return response["content"]
        
        return "No se pudo completar la tarea en el límite de iteraciones"

print("✅ FunctionCallingAgent implementado")

### Probar el Agente

In [ ]:
# Crear agente con todas las herramientas
agent = FunctionCallingAgent(
    tools=[weather_tool, calculator_tool, search_tool],
    llm_backend="simulated"
)

# Probar con diferentes queries
queries = [
    "¿Cuál es el clima en Madrid?",
    "Calcula cuánto es 25 * 37 + 100",
    "Busca información sobre inteligencia artificial"
]

for query in queries:
    result = agent.run(query, verbose=True)
    print("\n" + "="*70 + "\n")

## 5. OpenAI Function Calling (Real API)

Ejemplo de uso con API real de OpenAI:

In [ ]:
# Ejemplo con OpenAI real (requiere API key)
if OPENAI_AVAILABLE and os.getenv("OPENAI_API_KEY"):
    # Crear agente con OpenAI
    openai_agent = FunctionCallingAgent(
        tools=[weather_tool, calculator_tool],
        llm_backend="openai"
    )
    
    # Probar
    # result = openai_agent.run("What's the weather in London and how much is 15% of 340?")
    # print(result)
    
    print("✅ OpenAI agent configurado (descomenta para probar)")
else:
    print("⚠️  OpenAI no disponible o API key no configurada")

## 6. Seguridad y Sandboxing

### Consideraciones de Seguridad

1. **Input Validation**: Siempre validar argumentos contra schema
2. **Sandboxing**: Ejecutar código en entorno aislado
3. **Rate Limiting**: Limitar llamadas a APIs externas
4. **Access Control**: Verificar permisos antes de ejecutar
5. **Logging**: Registrar todas las ejecuciones para auditoría

In [ ]:
# Ejemplo de sandboxing para ejecución de código
import subprocess
import tempfile

def execute_python_sandboxed(code: str, timeout: int = 5) -> Dict:
    """
    Ejecuta código Python en proceso separado con timeout.
    
    NOTA: Esto es una simplificación educativa.
    En producción usar containers (Docker) o servicios especializados.
    """
    try:
        # Crear archivo temporal
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(code)
            temp_file = f.name
        
        # Ejecutar con timeout
        result = subprocess.run(
            ['python', temp_file],
            capture_output=True,
            text=True,
            timeout=timeout
        )
        
        return {
            "stdout": result.stdout,
            "stderr": result.stderr,
            "returncode": result.returncode
        }
    
    except subprocess.TimeoutExpired:
        return {"error": "Timeout: código tomó más de 5 segundos"}
    except Exception as e:
        return {"error": str(e)}

# Ejemplo de uso
safe_code = "print('Hello from sandboxed environment!')"
result = execute_python_sandboxed(safe_code)
print("Sandboxed execution:")
print(result)

## 7. Ejercicios

### 🟢 Ejercicio 1: Crear Nueva Herramienta

In [ ]:
def ejercicio_1_email_tool():
    """
    Objetivo: Crear una herramienta de envío de email (simulado)
    
    Instrucciones:
    1. Define función send_email(to, subject, body)
    2. Crea Tool con JSON schema apropiado
    3. Agrega al agente
    4. Prueba con: "Envía un email a john@example.com sobre la reunión"
    """
    # TODO: Tu código aquí
    pass

# ejercicio_1_email_tool()

### 🟡 Ejercicio 2: Tool Composition

In [ ]:
def ejercicio_2_tool_composition():
    """
    Objetivo: Crear tarea que requiere múltiples herramientas
    
    Tarea: "Busca el clima en Madrid y Londres, compara las temperaturas y 
            dime cuál es más cálida y por cuántos grados"
    
    Requiere:
    - get_weather (2 veces)
    - calculate (para restar)
    - Razonamiento para sintetizar respuesta
    """
    # TODO: Tu código aquí
    pass

# ejercicio_2_tool_composition()

### 🔴 Ejercicio 3: API Integration Real

In [ ]:
def ejercicio_3_real_api():
    """
    Objetivo: Integrar API real (OpenWeatherMap, SerpAPI, etc.)
    
    Instrucciones:
    1. Obtén API key gratuita de OpenWeatherMap
    2. Implementa get_weather real usando requests
    3. Maneja errores (rate limits, network issues)
    4. Agrega caching para reducir llamadas
    5. Prueba con ciudades reales
    """
    # TODO: Tu código aquí
    # Pista: https://openweathermap.org/api
    pass

# Este ejercicio requiere API key externa

## 8. Resumen y Recursos

### 📚 Resumen

- **Function Calling**: Permite a LLMs usar herramientas externas de forma estructurada
- **JSON Schema**: Estándar para definir parámetros de herramientas
- **Tool Components**: Name, Description, Parameters, Implementation
- **Seguridad**: Validación, sandboxing, rate limiting son esenciales
- **APIs Reales**: OpenAI y Anthropic tienen function calling built-in
- **Tool Composition**: Tareas complejas requieren múltiples herramientas

### 🔗 Recursos Adicionales

#### 📄 Papers

1. **"Toolformer: Language Models Can Teach Themselves to Use Tools"** (Schick et al., 2023)
   - [https://arxiv.org/abs/2302.04761](https://arxiv.org/abs/2302.04761)
   
2. **"Gorilla: Large Language Model Connected with Massive APIs"** (Patil et al., 2023)
   - [https://arxiv.org/abs/2305.15334](https://arxiv.org/abs/2305.15334)

#### 💻 Documentación

- **OpenAI Function Calling**: [Docs](https://platform.openai.com/docs/guides/function-calling)
- **Anthropic Tool Use**: [Docs](https://docs.anthropic.com/claude/docs/tool-use)
- **JSON Schema**: [Spec](https://json-schema.org/)

### ➡️ Próximo Paso

**[➡️ Ir al Notebook 05: Memory Systems](05-memory-systems.ipynb)**

---

<div align="center">

### Respuesta a la Pregunta Guía

*¿Cómo dar acceso seguro a herramientas externas?*

**Respuesta:**
1. **Validación**: JSON Schema garantiza argumentos correctos
2. **Sandboxing**: Aislar ejecución de código no confiable
3. **Permisos**: Control de acceso explícito por herramienta
4. **Auditoría**: Logging de todas las ejecuciones
5. **Rate Limiting**: Prevenir abuso de APIs

Function Calling es seguro cuando se implementa correctamente.

</div>